In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("PracticePyspark 20261020").getOrCreate()

In [0]:
orders_data = [
    (1001, 101, 2500),
    (1002, 102, 1800),
    (1003, 106, 3200),
    (1004, 103, 1500),
    (1005, 108, 4500),
    (1006, 105, 2100),
    (1007, 109, 900)
]

orders_columns = ["order_id", "customer_id", "amount"]

orders_df = spark.createDataFrame(
    orders_data, orders_columns
)

orders_df.show()

+--------+-----------+------+
|order_id|customer_id|amount|
+--------+-----------+------+
|    1001|        101|  2500|
|    1002|        102|  1800|
|    1003|        106|  3200|
|    1004|        103|  1500|
|    1005|        108|  4500|
|    1006|        105|  2100|
|    1007|        109|   900|
+--------+-----------+------+



In [0]:
customers_data = [
    (101, "Rahul", "Ahmedabad"),
    (102, "Priya", "Gandhinagar"),
    (103, "Amit", "Surat"),
    (104, "Neha", "Vadodara"),
    (105, "Rohan", "Rajkot")
]

customers_columns = ["customer_id", "customer_name", "city"]

customers_df = spark.createDataFrame(
    customers_data, customers_columns
)

customers_df.show()

+-----------+-------------+-----------+
|customer_id|customer_name|       city|
+-----------+-------------+-----------+
|        101|        Rahul|  Ahmedabad|
|        102|        Priya|Gandhinagar|
|        103|         Amit|      Surat|
|        104|         Neha|   Vadodara|
|        105|        Rohan|     Rajkot|
+-----------+-------------+-----------+



In [0]:
results = orders_df.join(customers_df , on = "customer_id" , how= "inner")

results.display()

customer_id,order_id,amount,customer_name,city
101,1001,2500,Rahul,Ahmedabad
102,1002,1800,Priya,Gandhinagar
103,1004,1500,Amit,Surat
105,1006,2100,Rohan,Rajkot


In [0]:
new_row = [(108,"test","Jaipur")]

new_test_df = spark.createDataFrame(new_row,("customer_id" , "customer_name" ,"city" ))

customers_df = customers_df.unionAll(new_test_df)

display(customers_df)

customer_id,customer_name,city
101,Rahul,Ahmedabad
102,Priya,Gandhinagar
103,Amit,Surat
104,Neha,Vadodara
105,Rohan,Rajkot
108,test,Jaipur
108,test,Jaipur


In [0]:
# to identify unmatched customer Id

unmatched_df = customers_df.join(orders_df,on="customer_id",how = "left_anti")
display(unmatched_df)

customer_id,customer_name,city
104,Neha,Vadodara


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

data = [
    (1, "Rahul", "IT", 50000),
    (2, "Priya", "IT", 70000),
    (3, "Amit", "IT", 70000),
    (4, "Neha", "HR", 40000),
    (5, "Rohan", "HR", 60000)
]

df = spark.createDataFrame(
    data, ["id", "name", "dept", "salary"]
)


Find the highest-paid employee in each department.

Find the top 2 salaries in each department.

Calculate each employee's salary difference from their department average.

Calculate a running salary total within each department.

In [0]:
w = Window.partitionBy("dept").orderBy(F.col("salary").desc())

#F gives you access to the functions inside pyspark.sql.functions.
results_df = df.withColumn("rnk", F.dense_rank().over(w)).filter(F.col("rnk") ==  1)

display(results_df)

id,name,dept,salary,rnk
5,Rohan,HR,60000,1
2,Priya,IT,70000,1
3,Amit,IT,70000,1


In [0]:
results_top2_df = df.withColumn("rnk",F.dense_rank().over(w)).filter(F.col("rnk") < 3)

display(results_top2_df)

id,name,dept,salary,rnk
5,Rohan,HR,60000,1
4,Neha,HR,40000,2
2,Priya,IT,70000,1
3,Amit,IT,70000,1
1,Rahul,IT,50000,2


In [0]:
%sql
--Calculate each employee's salary difference from their department average.
with dept_avg_salary as (
Select id, name, dept,salary , avg(salary) over (partition by dept) as department_avg from employees
)

select id, name, dept, salary, (salary - department_avg) as salarydiff from dept_avg_salary order by dept, salary



---------------------------------------------------------------------------
ParseException                            Traceback (most recent call last)
File <command-8655302126694404>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "--Calculate each employee's salary difference from their department average.\nwith dept_avg_salary as (\nSelect id, name, dept,salary , avg(salary) over (partition by salary, group by dept) as department_avg from employees\n)\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, Fa

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Calculate average salary for each department
w = Window.partitionBy("dept")

avg_salary_diff_df = (
    df.withColumn(
        "avg_salary_dept",
        F.avg("salary").over(w)
    )
    .withColumn(
        "avg_salary_diff",
        F.col("salary") - F.col("avg_salary_dept")
    )
)

display(avg_salary_diff_df)

id,name,dept,salary,avg_salary_dept
1,Rahul,IT,50000,63333.333333333336
2,Priya,IT,70000,63333.333333333336
3,Amit,IT,70000,63333.333333333336
4,Neha,HR,40000,50000.0
5,Rohan,HR,60000,50000.0


id,name,dept,salary,avg_salary_dept,avg_salary_diff
1,Rahul,IT,50000,63333.333333333336,-13333.333333333336
2,Priya,IT,70000,63333.333333333336,6666.666666666664
3,Amit,IT,70000,63333.333333333336,6666.666666666664
4,Neha,HR,40000,50000.0,-10000.0
5,Rohan,HR,60000,50000.0,10000.0


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = (
    Window.partitionBy("dept")
    .orderBy("salary")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

running_total_df = df.withColumn(
    "SalarySum",
    F.sum("salary").over(w)
)

display(running_total_df)

id,name,dept,salary,SalarySum
4,Neha,HR,40000,40000
5,Rohan,HR,60000,100000
1,Rahul,IT,50000,50000
2,Priya,IT,70000,120000
3,Amit,IT,70000,190000
